# Snapshot Comparison

Compare two snapshots (default: baseline vs latest) to measure intervention impact on build performance, dependencies, and critical path.

In [ ]:
from __future__ import annotations
import warnings
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
warnings.filterwarnings("ignore", category=FutureWarning)

SNAPSHOTS_DIR = Path("../../data/snapshots")
PROJECT_ROOT = Path("../..")

# Parameterise: which snapshots to compare
SNAPSHOT_A = "baseline"
SNAPSHOT_B = "latest"

from buildanalysis.snapshots import SnapshotManager
from buildanalysis.comparison import (
    compute_global_deltas, compute_target_deltas,
    compute_edge_deltas, compute_critical_path_comparison,
)

sm = SnapshotManager(SNAPSHOTS_DIR)

plt.rcParams.update({"figure.figsize": (12, 6), "figure.dpi": 100, "axes.titlesize": 14})
sns.set_theme(style="whitegrid", palette="muted")

In [ ]:
# Resolve snapshot labels
all_snaps = sm.list_snapshots()
print(f"Available snapshots: {len(all_snaps)}")
for snap in all_snaps:
    print(f"  - {snap.label}")

snap_a_meta = next((s for s in all_snaps if s.label.startswith(SNAPSHOT_A)), None)
snap_b_meta = None

if SNAPSHOT_B == "latest":
    snap_b_meta = all_snaps[-1] if all_snaps else None
else:
    snap_b_meta = next((s for s in all_snaps if s.label.startswith(SNAPSHOT_B)), None)

if not snap_a_meta or not snap_b_meta:
    print(f"\nCould not find snapshots: {SNAPSHOT_A} or {SNAPSHOT_B}")
    print("Proceeding with available data...")
else:
    print(f"\nComparing: {snap_a_meta.label} vs {snap_b_meta.label}")

In [ ]:
# Load datasets using SnapshotManager.load_pair
if snap_a_meta and snap_b_meta:
    ds_a, ds_b = sm.load_pair(snap_a_meta.label, snap_b_meta.label)
    print(f"Loaded snapshot A: {snap_a_meta.label}")
    print(f"Loaded snapshot B: {snap_b_meta.label}")
else:
    print("Cannot proceed without both snapshots")

## 1. Snapshot Metadata

In [ ]:
if snap_a_meta and snap_b_meta:
    print("\nSNAPSHOT METADATA")
    print("=" * 100)

    metadata_comparison = pd.DataFrame([
        {
            "Property": "Label",
            "Snapshot A (Baseline)": snap_a_meta.label,
            "Snapshot B (Latest)": snap_b_meta.label,
        },
        {
            "Property": "Date",
            "Snapshot A (Baseline)": snap_a_meta.date,
            "Snapshot B (Latest)": snap_b_meta.date,
        },
        {
            "Property": "Git Ref",
            "Snapshot A (Baseline)": snap_a_meta.git_ref,
            "Snapshot B (Latest)": snap_b_meta.git_ref,
        },
        {
            "Property": "Compiler",
            "Snapshot A (Baseline)": snap_a_meta.compiler,
            "Snapshot B (Latest)": snap_b_meta.compiler,
        },
        {
            "Property": "Notes",
            "Snapshot A (Baseline)": snap_a_meta.notes,
            "Snapshot B (Latest)": snap_b_meta.notes,
        },
    ])

    print(metadata_comparison.to_string(index=False))
else:
    print("Cannot display metadata without both snapshots")

## 2. Global Metric Deltas

In [ ]:
if snap_a_meta and snap_b_meta:
    # Use compute_global_deltas for richer comparison (10 metrics with improved/regressed flags)
    delta_df = compute_global_deltas(ds_a, ds_b)

    print("\nGLOBAL METRIC DELTAS")
    print("=" * 100)
    print(delta_df.to_string(index=False))

    # Plot
    numeric_deltas = delta_df[delta_df["delta_pct"].notna()].copy()
    if len(numeric_deltas) > 0:
        fig, axes = plt.subplots(1, 2, figsize=(14, 5))

        colors = ["#2ca02c" if x < 0 else "#d62728" for x in numeric_deltas["delta_pct"]]
        axes[0].barh(range(len(numeric_deltas)), numeric_deltas["delta_pct"], color=colors)
        axes[0].set_yticks(range(len(numeric_deltas)))
        axes[0].set_yticklabels(numeric_deltas["metric"])
        axes[0].axvline(x=0, color="black", linestyle="-", linewidth=0.8)
        axes[0].set_xlabel("Change (%)")
        axes[0].set_title("Global Metric Deltas (%)")

        x = np.arange(len(numeric_deltas))
        width = 0.35
        axes[1].bar(x - width/2, numeric_deltas["before"], width, label="Baseline", alpha=0.8)
        axes[1].bar(x + width/2, numeric_deltas["after"], width, label="Latest", alpha=0.8)
        axes[1].set_ylabel("Value")
        axes[1].set_title("Absolute Metrics")
        axes[1].set_xticks(x)
        axes[1].set_xticklabels(numeric_deltas["metric"], rotation=45, ha="right")
        axes[1].legend()

        plt.tight_layout()
        plt.show()
else:
    print("Cannot compute global deltas without both snapshots")

## 3. Target-Level Deltas

In [ ]:
if snap_a_meta and snap_b_meta:
    # Use compute_target_deltas for richer comparison (status flags: improved/regressed/new/removed)
    target_comparison = compute_target_deltas(ds_a, ds_b)

    print("\nTARGET-LEVEL DELTAS")
    print("=" * 100)

    status_counts = target_comparison["status"].value_counts()
    for status, count in status_counts.items():
        print(f"  {status}: {count}")

    improvers = target_comparison[target_comparison["status"] == "improved"].nsmallest(20, "build_time_delta_ms")
    regressors = target_comparison[target_comparison["status"] == "regressed"].nlargest(20, "build_time_delta_ms")

    if len(improvers) > 0:
        print(f"\nTop 20 Improvers:")
        display_cols = [c for c in ["cmake_target", "build_time_before_ms", "build_time_after_ms",
                                     "build_time_delta_ms", "build_time_delta_pct", "status"]
                       if c in improvers.columns]
        print(improvers[display_cols].to_string(index=False))

    if len(regressors) > 0:
        print(f"\nTop 20 Regressors:")
        display_cols = [c for c in ["cmake_target", "build_time_before_ms", "build_time_after_ms",
                                     "build_time_delta_ms", "build_time_delta_pct", "status"]
                       if c in regressors.columns]
        print(regressors[display_cols].to_string(index=False))

    # New / removed targets
    new_count = (target_comparison["status"] == "new").sum()
    removed_count = (target_comparison["status"] == "removed").sum()
    print(f"\nNew targets: {new_count}")
    print(f"Removed targets: {removed_count}")

    # Plot
    top_changes = pd.concat([improvers.head(10), regressors.head(10)]).sort_values("build_time_delta_ms")
    if len(top_changes) > 0:
        fig, ax = plt.subplots(figsize=(12, 8))
        colors = ["#2ca02c" if x < 0 else "#d62728" for x in top_changes["build_time_delta_ms"]]
        ax.barh(range(len(top_changes)), top_changes["build_time_delta_ms"], color=colors)
        ax.set_yticks(range(len(top_changes)))
        ax.set_yticklabels(top_changes["cmake_target"], fontsize=9)
        ax.axvline(x=0, color="black", linestyle="-", linewidth=0.8)
        ax.set_xlabel("Build Time Delta (ms)")
        ax.set_title("Top 10 Improvers & Regressors")
        plt.tight_layout()
        plt.show()
else:
    print("Cannot compute target deltas without both snapshots")

## 4. Dependency Changes

In [ ]:
if snap_a_meta and snap_b_meta:
    # Use compute_edge_deltas for structured comparison
    edge_deltas = compute_edge_deltas(ds_a, ds_b)

    print("\nDEPENDENCY CHANGES")
    print("=" * 80)
    print(f"Added edges: {edge_deltas['added_count']}")
    print(f"Removed edges: {edge_deltas['removed_count']}")
    print(f"Unchanged edges: {edge_deltas['edges_unchanged']}")
    print(f"Total change: {edge_deltas['added_count'] - edge_deltas['removed_count']:+d} edges")

    if len(edge_deltas["edges_added"]) > 0:
        print(f"\nSample added edges (first 10):")
        for _, row in edge_deltas["edges_added"].head(10).iterrows():
            print(f"  {row['source_target']} -> {row['dest_target']}")

    if len(edge_deltas["edges_removed"]) > 0:
        print(f"\nSample removed edges (first 10):")
        for _, row in edge_deltas["edges_removed"].head(10).iterrows():
            print(f"  {row['source_target']} -> {row['dest_target']}")
else:
    print("Cannot compute dependency changes without both snapshots")

## 5. Critical Path Comparison

In [ ]:
if snap_a_meta and snap_b_meta:
    from buildanalysis.graph import build_dependency_graph

    # Build graphs for critical path comparison
    bg_a = build_dependency_graph(ds_a.target_metrics, ds_a.edge_list)
    bg_b = build_dependency_graph(ds_b.target_metrics, ds_b.edge_list)

    # Use compute_critical_path_comparison for proper DP-based critical path analysis
    try:
        cp_comparison = compute_critical_path_comparison(ds_a, ds_b, bg_a, bg_b)

        time_a_ms = cp_comparison["before_time_s"] * 1000
        time_b_ms = cp_comparison["after_time_s"] * 1000
        delta_ms = cp_comparison["delta_s"] * 1000
        delta_pct = (delta_ms / time_a_ms * 100) if time_a_ms > 0 else 0.0

        print("\nCRITICAL PATH COMPARISON")
        print("=" * 80)
        print(f"Baseline critical path: {time_a_ms:.0f} ms ({len(cp_comparison['before_critical_path'])} targets)")
        print(f"Latest critical path: {time_b_ms:.0f} ms ({len(cp_comparison['after_critical_path'])} targets)")
        print(f"Delta: {delta_ms:+.0f} ms ({delta_pct:+.1f}%)")

        if delta_ms < 0:
            print(f"\nCritical path improved by {abs(delta_ms):.0f} ms")
        elif delta_ms > 0:
            print(f"\nCritical path regressed by {delta_ms:.0f} ms")

        if cp_comparison.get("targets_added_to_path"):
            print(f"\nTargets added to critical path: {', '.join(cp_comparison['targets_added_to_path'][:10])}")
        if cp_comparison.get("targets_removed_from_path"):
            print(f"Targets removed from critical path: {', '.join(cp_comparison['targets_removed_from_path'][:10])}")
    except Exception as e:
        print(f"\nCould not compute critical path comparison: {e}")
else:
    print("Cannot compute critical path without both snapshots")

## 6. Intervention Effectiveness

In [ ]:
if snap_a_meta and snap_b_meta:
    interventions = snap_b_meta.interventions_applied if hasattr(snap_b_meta, "interventions_applied") else []

    print("\nINTERVENTION EFFECTIVENESS")
    print("=" * 80)

    if isinstance(interventions, list) and len(interventions) > 0:
        print(f"Interventions applied in {snap_b_meta.label}:")
        for intervention in interventions:
            print(f"  - {intervention}")

        print(f"\nObserved improvements:")
        for idx, row in delta_df.iterrows():
            if row["delta_pct"] < 0:
                print(f"  ✓ {row['metric']}: {row['delta_pct']:.1f}% improvement")

        print(f"\nObserved regressions:")
        for idx, row in delta_df.iterrows():
            if row["delta_pct"] > 0:
                print(f"  ✗ {row['metric']}: {row['delta_pct']:.1f}% regression")
    else:
        print("No interventions recorded for this snapshot.")
        print("\nOverall metric changes:")
        for idx, row in delta_df.iterrows():
            if row["delta_pct"] < 0:
                print(f"  ✓ {row['metric']}: {row['delta_pct']:.1f}% improvement")
            elif row["delta_pct"] > 0:
                print(f"  ✗ {row['metric']}: {row['delta_pct']:.1f}% regression")
else:
    print("Cannot assess intervention effectiveness without both snapshots")

## 7. Summary

In [ ]:
if snap_a_meta and snap_b_meta:
    print("\n" + "=" * 80)
    print("SNAPSHOT COMPARISON SUMMARY")
    print("=" * 80)

    # Overall assessment
    improved_count = sum(1 for _, row in delta_df.iterrows() if row["delta_pct"] < 0)
    regressed_count = sum(1 for _, row in delta_df.iterrows() if row["delta_pct"] > 0)

    if improved_count > regressed_count:
        print(f"\n✓ BUILD IMPROVED: {improved_count} metrics improved, {regressed_count} regressed")
    elif regressed_count > improved_count:
        print(f"\n✗ BUILD REGRESSED: {regressed_count} metrics regressed, {improved_count} improved")
    else:
        print(f"\n= NO CLEAR CHANGE: {improved_count} metrics improved, {regressed_count} regressed")

    # Specific callouts
    build_time_delta = delta_df[delta_df["metric"] == "total_build_time_ms"]["delta_pct"].values
    if len(build_time_delta) > 0:
        build_pct = build_time_delta[0]
        if build_pct < 0:
            print(f"\nBuild time improved by {abs(build_pct):.1f}%")
        elif build_pct > 0:
            print(f"\nBuild time regressed by {build_pct:.1f}%")

    print(f"\nComparison: {snap_a_meta.label} → {snap_b_meta.label}")
    print("=" * 80)
else:
    print("Cannot generate summary without both snapshots")